## PostGIS Integration — Loading Wildfire Data & Building Spatial Queries

**Data:** CAL FIRE historical fire perimeters (`fire24_1.gdb`, 22,810 records) + NASA FIRMS active fire detections (VIIRS S-NPP, California, 2025)

**Database:** PostgreSQL + PostGIS 3.6 (`wildfire_db`, native Windows install)

**What this notebook does:**
- Connects to `wildfire_db` via SQLAlchemy engine and psycopg2 — the two Python interfaces to PostgreSQL
- Loads CAL FIRE fire perimeters from `fire24_1.gdb` into PostGIS table `fire_perimeters` using `gdf.to_postgis()` and creates a GiST spatial index for fast querying
- Downloads NASA FIRMS active fire detection CSV and loads it into PostGIS table `firms_fires`, creating point geometries from lat/lon columns using `ST_MakePoint` and a GiST spatial index
- Writes core spatial queries that will become FastAPI endpoints: `ST_Intersects` (FIRMS detections inside a fire perimeter), `ST_DWithin` (fire perimeters within N km of a coordinate), and top fires by acreage within a radius
- Builds a Python function `get_nearby_fires(lat, lon, radius_km)` that connects to PostGIS, runs a parameterized `ST_DWithin` query, and returns results as a GeoDataFrame via `gpd.read_postgis()`
- Visualizes query results with matplotlib to verify spatial correctness

In [5]:
from dotenv import load_dotenv
import os

load_dotenv()

password = os.getenv("DB_PASSWORD")
user = os.getenv("DB_USER")
host = os.getenv("DB_HOST")
dbname = os.getenv("DB_NAME")


In [6]:
import geopandas as gpd
import sqlalchemy as sa
from sqlalchemy import create_engine

In [7]:
# create geodataframe
gdf = gpd.read_file("../data/raw/fire24_1.gdb", layer="firep24_1")

f:\GeoPandas Projects\ca-wildfire-project\venv\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: organizePolygons() received a polygon with more than 100 parts. The processing may be really slow.  You can skip the processing by setting METHOD=SKIP, or only make it analyze counter-clock wise parts by setting METHOD=ONLY_CCW if you can assume that the outline of holes is counter-clock wise defined
  return ogr_read(


In [8]:
#Create SQLAlchemy engine
engine = create_engine(f"postgresql+psycopg2://{user}:{password}@{host}/{dbname}")


In [9]:
# Rerun this cell only if it is first time running the code, or if you want to replace the existing table in the database.
# gdf.to_postgis("fire_perimeters", engine, if_exists="replace")

In [10]:
#Verify data was written to PostGIS
import psycopg2
conn = psycopg2.connect(host=host, dbname=dbname, user=user, password=password)
cursor = conn.cursor()
cursor.execute("select * from fire_perimeters limit 5;")
results = cursor.fetchall()
print(results)

[(2025.0, 'CA', 'CDF', 'LDF', 'PALISADES', '00000738', '{A7EA5D21-F882-44B8-BF64-44AB11059DC1}', datetime.datetime(2025, 1, 6, 16, 0, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=57600))), datetime.datetime(2025, 1, 30, 16, 0, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=57600))), 7.0, 14, None, None, 1.0, 23448.883, None, None, 116028.19734856366, 94894264.13284907, '...'), (2025.0, 'CA', 'CDF', 'LAC', 'EATON', '00009087', '{72660ADC-B5EF-4D96-A33F-B4EA3740A4E3}', datetime.datetime(2025, 1, 7, 16, 0, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=57600))), datetime.datetime(2025, 1, 30, 16, 0, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=57600))), 7.0, 14, None, None, 1.0, 14056.261, None, None, 86677.54505602142, 56883667.35581151, '...'), (2025.0, 'CA', 'CDF', 'ANF', 'HUGHES', '00250270', '{994072D2-E154-434A-BB95-6F6C94C40829}', datetime.datetime(2025, 1, 21, 16, 0, tzinfo=datetime.timezone(datetime.timedelta(days=-1, se

In [11]:
# Only run this cell to create a spatial index on the geometry column if you haven't already, or if you want to recreate the index.

# cursor.execute("CREATE INDEX idx_fire_perimeters_geom ON fire_perimeters USING GIST(geometry);")
# conn.commit()


### Loading NASA FIRMS data
Example data downloaded was from NASA FIRMS for date of Malibu Fires (01/01/2025 - 03/31/2025)
Source: https://firms.modaps.eosdis.nasa.gov/

In [12]:
import pandas as pd

In [13]:
firms_df = pd.read_csv("../data/raw/NASA_FIRMS/fire_archive_SV-C2_738489.csv")
firms_df.head()


,latitude,longitude,brightness,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_t31,frp,daynight,type
0,40.02568,-74.25027,297.86,0.48,0.40,2025-01-01,633,N,VIIRS,n,2,277.77,1.02,N,2
1,37.76586,-76.51810,295.04,0.38,0.43,2025-01-01,633,N,VIIRS,n,2,279.34,0.45,N,0
2,38.33913,-75.84859,301.90,0.54,0.42,2025-01-01,633,N,VIIRS,n,2,278.31,0.65,N,0
3,40.17477,-75.89853,298.00,0.57,0.43,2025-01-01,633,N,VIIRS,n,2,276.34,1.05,N,2
4,38.63375,-75.75109,305.35,0.54,0.42,2025-01-01,633,N,VIIRS,n,2,277.13,1.26,N,0


In [ ]:
# Write to PosgreSQL
# firms_df.to_sql("firms_fires", engine, if_exists="replace", index=False)

f:\GeoPandas Projects\ca-wildfire-project\venv\Lib\site-packages\pandas\io\sql.py:2071: SAWarning: Did not recognize type 'geometry' of column 'geom'
  self.meta.reflect(


848

In [ ]:
cursor.execute("SELECT * FROM firms_fires LIMIT 10;")
results = cursor.fetchall()
print(results)

[(40.02568, -74.25027, 297.86, 0.48, 0.4, '2025-01-01', 633, 'N', 'VIIRS', 'n', 2, 277.77, 1.02, 'N', 2), (37.76586, -76.5181, 295.04, 0.38, 0.43, '2025-01-01', 633, 'N', 'VIIRS', 'n', 2, 279.34, 0.45, 'N', 0), (38.33913, -75.84859, 301.9, 0.54, 0.42, '2025-01-01', 633, 'N', 'VIIRS', 'n', 2, 278.31, 0.65, 'N', 0), (40.17477, -75.89853, 298.0, 0.57, 0.43, '2025-01-01', 633, 'N', 'VIIRS', 'n', 2, 276.34, 1.05, 'N', 2), (38.63375, -75.75109, 305.35, 0.54, 0.42, '2025-01-01', 633, 'N', 'VIIRS', 'n', 2, 277.13, 1.26, 'N', 0), (39.42359, -74.54429, 299.24, 0.49, 0.4, '2025-01-01', 633, 'N', 'VIIRS', 'n', 2, 279.14, 0.59, 'N', 2), (40.17414, -75.89795, 304.62, 0.57, 0.43, '2025-01-01', 633, 'N', 'VIIRS', 'n', 2, 276.35, 1.27, 'N', 2), (34.34097, -77.89012, 298.24, 0.41, 0.45, '2025-01-01', 634, 'N', 'VIIRS', 'n', 2, 284.93, 0.23, 'N', 0), (34.48626, -81.63371, 314.37, 0.33, 0.56, '2025-01-01', 634, 'N', 'VIIRS', 'n', 2, 279.66, 1.15, 'N', 0), (35.13957, -82.09725, 308.31, 0.36, 0.57, '2025-01

In [ ]:
# Integrate geometry column for PostGIS to run ST_Dwithin and Intersect queries, only perform once
# cursor.execute("ALTER TABLE firms_fires ADD COLUMN geom geometry(Point, 4326);")
# conn.commit()

# cursor.execute("UPDATE firms_fires SET geom = ST_SetSRID(ST_MakePoint(Longitude, Latitude), 4326);")
# conn.commit()

# cursor.execute("CREATE INDEX idx_firms_geom ON firms_fires USING GIST(geom);")
# conn.commit()


In [17]:
cursor.execute("SELECT latitude, longitude, geom FROM firms_fires LIMIT 3;")
results = cursor.fetchall()
print(results)

[(40.02568, -74.25027, '0101000020E6100000E04A766C049052C0A514747B49034440'), (37.76586, -76.5181, '0101000020E6100000B003E78C282153C044A852B307E24240'), (38.33913, -75.84859, '0101000020E61000009C6D6E4C4FF652C0D68BA19C682B4340')]


### Query to find which FIRMS satellite detections fell inside LA fire perimeters

In [18]:
conn.rollback()

In [19]:
# Use this query to identify rows in the fires_perimeters table
# cursor.execute("SELECT column_name FROM information_schema.columns WHERE table_name = 'fire_perimeters';")
# for row in cursor.fetchall():
#     print(row)

In [20]:
#Identify how many FIRMS detections fall within the palisades fire perimeter
cursor.execute('SELECT f.* '
'FROM fire_perimeters as p, firms_fires as f '
'WHERE ST_Intersects(ST_Transform(p.geometry, 4326), f.geom) ' # Match CRS of fire_perimeters with Firms_fires
'AND p."FIRE_NAME" = \'PALISADES\' '
'AND p."YEAR_" = 2025;')
results = cursor.fetchall()
print(len(results))
print(results)


931
[(34.07848, -118.67812, 308.01, 0.68, 0.74, '2025-01-09', 1047, 'N', 'VIIRS', 'n', 2, 283.62, 3.75, 'N', 0, '0101000020E6100000DEB06D5166AB5DC0EEB1F4A10B0A4140'), (34.07856, -118.67613, 338.55, 0.38, 0.59, '2025-01-10', 1028, 'N', 'VIIRS', 'n', 2, 287.14, 3.61, 'N', 0, '0101000020E61000000C76C3B645AB5DC00B630B410E0A4140'), (34.07859, -118.67194, 337.17, 0.42, 0.61, '2025-01-10', 2008, 'N', 'VIIRS', 'n', 2, 295.37, 9.25, 'D', 0, '0101000020E6100000F437A11001AB5DC075E5B33C0F0A4140'), (34.07871, -118.66563, 327.07, 0.42, 0.38, '2025-01-08', 2045, 'N', 'VIIRS', 'n', 2, 291.75, 11.53, 'D', 0, '0101000020E6100000224F92AE99AA5DC020EF552B130A4140'), (34.08342, -118.63853, 341.6, 0.42, 0.38, '2025-01-08', 2045, 'N', 'VIIRS', 'n', 2, 298.73, 98.2, 'D', 0, '0101000020E6100000F4E0EEACDDA85DC089EAAD81AD0A4140'), (34.08064, -118.63467, 326.86, 0.48, 0.4, '2025-01-08', 926, 'N', 'VIIRS', 'n', 2, 281.81, 1.66, 'N', 0, '0101000020E61000002670EB6E9EA85DC0EE5F5969520A4140'), (34.08129, -118.63994, 30

In [21]:
#Identify all the fire perimeters within 50km of Santa Monica
conn.rollback()
cursor.execute('SELECT p.* '
        'FROM fire_perimeters as p '
        'WHERE ST_DWithin(ST_Transform(p.geometry, 4326)::geography, ST_SetSRID(ST_MakePoint(-118.4912, 34.0007), 4326)::geography, 50000) ')

results = cursor.fetchall()
print(f"{len(results)} fire perimeters found within 50km of Santa Monica")
for row in results[:5]:
    print(row)
   

1471 fire perimeters found within 50km of Santa Monica
(2025.0, 'CA', 'CDF', 'LDF', 'PALISADES', '00000738', '{A7EA5D21-F882-44B8-BF64-44AB11059DC1}', datetime.datetime(2025, 1, 6, 16, 0, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=57600))), datetime.datetime(2025, 1, 30, 16, 0, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=57600))), 7.0, 14, None, None, 1.0, 23448.883, None, None, 116028.19734856366, 94894264.13284907, '...')
(2025.0, 'CA', 'CDF', 'LAC', 'EATON', '00009087', '{72660ADC-B5EF-4D96-A33F-B4EA3740A4E3}', datetime.datetime(2025, 1, 7, 16, 0, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=57600))), datetime.datetime(2025, 1, 30, 16, 0, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=57600))), 7.0, 14, None, None, 1.0, 14056.261, None, None, 86677.54505602142, 56883667.35581151, '...')
(2025.0, 'CA', 'CCO', 'VNC', 'KENNETH', '00003155', '{842FB37B-7AC8-4700-BB9C-028BF753D149}', datetime.datetime(2025, 1, 8, 16, 0, tzi

In [22]:
# Identify top ten largest fires by acreage within 50km of Santa Monica
conn.rollback()
cursor.execute('SELECT p."FIRE_NAME", p."GIS_ACRES", p."YEAR_" '
        'FROM fire_perimeters as p '
        'WHERE ST_DWithin(ST_Transform(p.geometry, 4326)::geography, ST_SetSRID(ST_MakePoint(-118.4912, 34.0007), 4326)::geography, 50000) '
        'order by p."GIS_ACRES" desc limit 10;')
results = cursor.fetchall()
print(results)

[('STATION', 160833.11, 2009.0), ('BOBCAT', 115997.98, 2020.0), ('CLAMPITT', 115537.42, 1970.0), ('SIMI', 107570.4, 2003.0), ('WOOLSEY', 89551.38, 2018.0), ('RAVENNA', 70796.41, 1919.0), (' ', 59468.883, 1878.0), ('RANCH', 58410.336, 2007.0), ('MILL/USFS', 51220.44, 1975.0), ('DAYTON CANYON', 43097.434, 1982.0)]


In [ ]:
def get_nearby_fires(lat, lon, radius_km):
    sql = f"""
        SELECT "FIRE_NAME", "YEAR_", "GIS_ACRES", geometry
        FROM fire_perimeters
        WHERE ST_DWithin(
            ST_Transform(geometry, 4326)::geography,
            ST_SetSRID(ST_MakePoint({lon}, {lat}), 4326)::geography,
            {radius_km * 1000}
        )
        ORDER BY "GIS_ACRES" DESC
    """
    return gpd.read_postgis(sql, engine, geom_col="geometry")

In [24]:
gdf = get_nearby_fires(34.01, -118.49, 50)
print(gdf.shape)
print(gdf.head())


(1499, 4)
  FIRE_NAME   YEAR_  GIS_ACRES  \
0   STATION  2009.0  160833.11   
1    BOBCAT  2020.0  115997.98   
2  CLAMPITT  1970.0  115537.42   
3      SIMI  2003.0  107570.40   
4   WOOLSEY  2018.0   89551.38   

                                            geometry  
0  MULTIPOLYGON (((177598.1 -395955.64, 177605.81...  
1  MULTIPOLYGON (((177089.189 -418181.228, 177090...  
2  MULTIPOLYGON (((96735.146 -410459.433, 96757.4...  
3  MULTIPOLYGON (((118370.407 -400934.968, 118409...  
4  MULTIPOLYGON (((111268.366 -443190.301, 111271...  
